# Classification & Logistic Regression
## From Predicting Numbers to Predicting Categories

### CE 315 - Junior Design

**Last time:** We learned how models learn using gradient descent to predict **continuous values** (house prices).

**Today:** We'll learn to predict **categories** (yes/no, pass/fail, spam/not spam) using **Logistic Regression**.

**Real Dataset:** Heart disease prediction 

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# Set style
sns.set_style('whitegrid')
np.random.seed(42)

print("Libraries loaded successfully")

## Part 1: What is Classification?

### Regression vs Classification

**Regression (what we've been doing):**
- Predict a **number**: house price, temperature, stock price
- Output is continuous: can be any value

**Classification (today):**
- Predict a **category**: yes/no, spam/ham, dog/cat/bird
- Output is discrete: belongs to one class

### Real-World Classification Problems

- **Email:** Spam or Not Spam?
- **Medical:** Disease or Healthy?
- **Finance:** Approve or Deny Loan?
- **Engineering:** Component Pass or Fail Quality Check?
- **Security:** Fraudulent or Legitimate Transaction?

### Types of Classification

1. **Binary Classification:** 2 classes (today's focus)
2. **Multi-class Classification:** 3+ classes (e.g., low/medium/high risk)
3. **Multi-label Classification:** Multiple categories at once

## Part 2: Loading Real Data - Heart Disease Prediction

**Dataset:** Cleveland Heart Disease Database

**Goal:** Predict if a patient has heart disease based on medical measurements.

**Features:**
- Age
- Sex (1 = male, 0 = female)
- Chest pain type (1-4)
- Resting blood pressure
- Cholesterol
- Fasting blood sugar > 120 mg/dl (1 = true)
- Resting ECG results
- Maximum heart rate achieved
- Exercise induced angina (1 = yes)
- ST depression
- Slope of peak exercise ST segment
- Number of major vessels colored by fluoroscopy
- Thalassemia (blood disorder)

**Target:**
- 0 = No heart disease
- 1 = Heart disease present

In [ ]:
# Load the heart disease dataset
# We'll create a simplified version for this demo

# Column names
column_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
                'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

# Load from UCI repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'


df = pd.read_csv(url, names=column_names, na_values='?')
print("Data loaded from UCI repository")

# Convert target to binary (0 or 1)
df['target'] = (df['target'] > 0).astype(int)


# Remove rows with missing values
df = df.dropna()

print(f"\n Dataset shape: {df.shape[0]} patients, {df.shape[1]} columns")
print(f"\n Target distribution:")
print(df['target'].value_counts())
print(f"\n   {df['target'].value_counts()[0]} patients WITHOUT heart disease")
print(f"   {df['target'].value_counts()[1]} patients WITH heart disease")

In [ ]:
# Look at the first few rows
df.head(30)

## Part 3: Exploratory Data Analysis

Let's explore the data to understand the problem better.

In [ ]:
# Visualize the target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
df['target'].value_counts().plot(kind='bar', ax=axes[0], color=['cyan', 'yellow'], alpha=0.7)
axes[0].set_xlabel('Heart Disease')
axes[0].set_ylabel('Number of Patients')
axes[0].set_title('Target Distribution')
axes[0].set_xticklabels(['No (0)', 'Yes (1)'], rotation=0)

# Pie chart
df['target'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                   colors=['cyan', 'yellow'], labels=['No Disease', 'Disease'])
axes[1].set_ylabel('')
axes[1].set_title('Target Percentage')

plt.tight_layout()
plt.show()


In [ ]:
# Compare features between patients with/without disease
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

features_to_plot = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
feature_names = ['Age', 'Resting BP', 'Cholesterol', 'Max Heart Rate', 'ST Depression', 'Vessels Colored']

for i, (feature, name) in enumerate(zip(features_to_plot, feature_names)):
    # Box plot comparing disease vs no disease
    df.boxplot(column=feature, by='target', ax=axes[i])
    axes[i].set_xlabel('Heart Disease (0=No, 1=Yes)')
    axes[i].set_ylabel(name)
    axes[i].set_title(f'{name} by Heart Disease')
    plt.sca(axes[i])
    plt.xticks([1, 2], ['No', 'Yes'])

plt.suptitle('Feature Distributions by Target', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("  Notice which features show clear differences between the two groups.")
print("   These will be most predictive.")

## Part 4: Why Not Use Linear Regression?

**Question:** We have data, we have a target (0 or 1). Can't we just use linear regression?

Let's try it and see what happens

In [ ]:
# Simple example with just age predicting disease
from sklearn.linear_model import LinearRegression

X_simple = df[['age']].values
y_simple = df['target'].values

# Fit linear regression
lin_model = LinearRegression()
lin_model.fit(X_simple, y_simple)

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 1, 1)
plt.scatter(X_simple, y_simple, alpha=0.5, s=50)
plt.plot(X_simple, lin_model.predict(X_simple), 'r-', linewidth=2, label='Linear Regression')
plt.xlabel('Age')
plt.ylabel('Heart Disease (0 or 1)')
plt.title('Linear Regression for Classification')
plt.axhline(y=0.5, color='green', linestyle='--', label='Decision Threshold')
plt.legend()
plt.ylim(-0.5, 1.5)



## Part 5: Introducing Logistic Regression

### The Key Idea: Sigmoid Function

**Problem:** Linear regression gives unbounded outputs (-∞ to +∞)

**Solution:** Squeeze the output between 0 and 1 using the **sigmoid function**!

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Where $z = mx + b$ (the linear part)

### Properties of Sigmoid:
- Output is ALWAYS between 0 and 1 
- Can be interpreted as probability 
- Smooth S-shaped curve 
- When z = 0, σ(z) = 0.5 (decision boundary)

In [ ]:
# Visualize the sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 100)
sig_z = sigmoid(z)

plt.figure(figsize=(12, 5))

# Sigmoid function
plt.subplot(1, 2, 1)
plt.plot(z, sig_z, 'b-', linewidth=3)
plt.axhline(y=0.5, color='r', linestyle='--', label='Decision Boundary (0.5)')
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.axhline(y=1, color='k', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.3)
plt.xlabel('z (linear combination: mx + b)', fontsize=12)
plt.ylabel('σ(z) - Probability', fontsize=12)
plt.title('The Sigmoid Function', fontsize=14, fontweight='bold')
plt.legend()

# Comparison
plt.subplot(1, 2, 2)
plt.plot(z, z, 'r--', linewidth=2, label='Linear: y = z', alpha=0.7)
plt.plot(z, sig_z, 'b-', linewidth=3, label='Sigmoid: σ(z)')
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.axhline(y=1, color='k', linestyle='-', alpha=0.3)
plt.fill_between(z, -2, 0, alpha=0.2, color='red', label='Invalid for probability')
plt.fill_between(z, 1, 3, alpha=0.2, color='red')
plt.grid(True, alpha=0.3)
plt.xlabel('z', fontsize=12)
plt.ylabel('Output', fontsize=12)
plt.title('Linear vs Sigmoid', fontsize=14, fontweight='bold')
plt.ylim(-0.5, 1.5)
plt.legend()

plt.tight_layout()
plt.show()

print("Key Points:")
print("• z → -∞  ⟹  σ(z) → 0  (definitely class 0)")
print("• z = 0   ⟹  σ(z) = 0.5  (uncertain, 50/50)")
print("• z → +∞  ⟹  σ(z) → 1  (definitely class 1)")
print("\n Output is ALWAYS a valid probability")

## Part 6: Building a Logistic Regression Model

### The Model:

$$P(y=1|x) = \sigma(mx + b) = \frac{1}{1 + e^{-(mx+b)}}$$

- **Input:** Features (age, cholesterol, etc.)
- **Linear part:** $z = mx + b$
- **Sigmoid:** Squeeze to [0,1]
- **Output:** Probability of heart disease

### Decision Rule:
- If P(disease) ≥ 0.5 → Predict **disease** (class 1)
- If P(disease) < 0.5 → Predict **no disease** (class 0)

In [ ]:
# Start with just 2 features for visualization
X_2d = df[['age', 'thalach']].values  # Age and max heart rate
y = df['target'].values

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_2d, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} patients")
print(f"Testing set:  {len(X_test)} patients")

In [ ]:
# Train logistic regression
log_model = LogisticRegression(random_state=42)
log_model.fit(X_train, y_train)

print(" Model trained!")
print(f"\nModel parameters:")
print(f"  Coefficient for age: {log_model.coef_[0][0]:.4f}")
print(f"  Coefficient for max heart rate: {log_model.coef_[0][1]:.4f}")
print(f"  Intercept: {log_model.intercept_[0]:.4f}")
print(f"\n Decision boundary equation:")
print(f"   z = {log_model.coef_[0][0]:.4f}×age + {log_model.coef_[0][1]:.4f}×max_hr + {log_model.intercept_[0]:.4f}")
print(f"   P(disease) = σ(z)")

In [ ]:
# Visualize the decision boundary
def plot_decision_boundary(X, y, model, title):
    # Create mesh
    h = 1  # step size in the mesh
    x_min, x_max = X[:, 0].min() - 5, X[:, 0].max() + 5
    y_min, y_max = X[:, 1].min() - 10, X[:, 1].max() + 10
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict for each point in mesh
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(12, 5))
    
    # Left: Decision boundary
    plt.subplot(1, 2, 1)
    plt.contourf(xx, yy, Z, levels=20, cmap='RdYlGn_r', alpha=0.6)
    plt.colorbar(label='P(Heart Disease)')
    
    # Plot decision boundary line (where p=0.5)
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=3)
    
    # Plot data points
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlGn_r', 
                         edgecolors='black', s=100, alpha=0.8)
    plt.xlabel('Age', fontsize=12)
    plt.ylabel('Max Heart Rate', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    
    # Right: Just the points with decision regions
    plt.subplot(1, 2, 2)
    plt.contourf(xx, yy, Z > 0.5, levels=1, colors=['lightgreen', 'lightcoral'], alpha=0.3)
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=3, 
                linestyles='--', label='Decision Boundary')
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlGn_r', 
                         edgecolors='black', s=100, alpha=0.8)
    plt.xlabel('Age', fontsize=12)
    plt.ylabel('Max Heart Rate', fontsize=12)
    plt.title('Decision Regions', fontsize=14, fontweight='bold')

    
    plt.tight_layout()
    plt.show()

# Plot for training data
plot_decision_boundary(X_train, y_train, log_model, 'Logistic Regression Decision Boundary')

print("\n Interpretation:")
print("• Green region: Model predicts NO disease (P < 0.5)")
print("• Red region: Model predicts disease (P ≥ 0.5)")
print("• Black line: Decision boundary (P = 0.5)")
print("• Dots: Actual patients (green = healthy, red = disease)")

## Part 7: Making Predictions

Logistic regression gives us TWO types of output:

1. **Probabilities** (between 0 and 1)
2. **Class predictions** (0 or 1)

Let's see both

In [ ]:
# Make predictions on test set
y_pred_proba = log_model.predict_proba(X_test)[:, 1]  # Probability of class 1
y_pred = log_model.predict(X_test)  # Actual class predictions

# Show some examples
print("Sample Predictions:\n")
print(f"{'Age':<5} {'Max HR':<8} {'P(Disease)':<12} {'Prediction':<12} {'Actual':<12} {'Correct':<8}")
print("="*70)

for i in range(30):
    age = X_test[i, 0]
    hr = X_test[i, 1]
    prob = y_pred_proba[i]
    pred = 'Disease' if y_pred[i] == 1 else 'No Disease'
    actual = 'Disease' if y_test[i] == 1 else 'No Disease'
    correct = 'Yes' if pred == actual else 'No'
    
    print(f"{age:<5.0f} {hr:<8.0f} {prob:<12.3f} {pred:<12} {actual:<12} {correct:<8}")

print("\n Notice:")
print("• Probabilities close to 0 or 1 = confident predictions")
print("• Probabilities around 0.5 = uncertain predictions")

## Part 8: Evaluating Classification Models

**New Problem:** For regression, we used MSE/MAE. What about classification?

### Classification Metrics:

1. **Accuracy:** What % did we get right?
2. **Confusion Matrix:** Where did we make mistakes?
3. **Precision:** Of those we said "disease", how many really had it?
4. **Recall:** Of those with disease, how many did we catch?
5. **F1-Score:** Balance between precision and recall

In [ ]:
# Calculate accuracy
train_accuracy = log_model.score(X_train, y_train)
test_accuracy = log_model.score(X_test, y_test)

print(f" Model Performance:\n")
print(f"Training Accuracy: {train_accuracy:.3f} ({train_accuracy*100:.1f}%)")
print(f"Testing Accuracy:  {test_accuracy:.3f} ({test_accuracy*100:.1f}%)")

# But accuracy alone doesn't tell the full story!

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')

# Add labels to each quadrant
plt.text(0.5, 0.25, 'True\nNegative', ha='center', va='center', fontsize=10, color='white')
plt.text(1.5, 0.25, 'False\nPositive', ha='center', va='center', fontsize=10, color='darkblue')
plt.text(0.5, 1.25, 'False\nNegative', ha='center', va='center', fontsize=10, color='darkblue')
plt.text(1.5, 1.25, 'True\nPositive', ha='center', va='center', fontsize=10, color='darkblue')

plt.tight_layout()
plt.show()

print("\n Reading the Confusion Matrix:\n")
print(f"True Negatives (TN):  {cm[0,0]} - Correctly predicted NO disease")
print(f"False Positives (FP): {cm[0,1]} - Incorrectly predicted disease (Type I error)")
print(f"False Negatives (FN): {cm[1,0]} - Missed disease cases (Type II error) ")
print(f"True Positives (TP):  {cm[1,1]} - Correctly predicted disease")

print("\n  Which error is worse?")
print("   For medical diagnosis: False Negatives are often MUCH worse!")
print("   (Missing a disease can be life-threatening)")

In [ ]:
# Detailed classification report
print("\n Detailed Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

print("\n Metrics Explained:\n")
print("Precision = TP / (TP + FP)")
print("  → Of all patients we said had disease, what % actually did?")
print("  → High precision = few false alarms\n")

print("Recall = TP / (TP + FN)")
print("  → Of all patients with disease, what % did we catch?")
print("  → High recall = don't miss many cases\n")

print("F1-Score = 2 × (Precision × Recall) / (Precision + Recall)")
print("  → Harmonic mean of precision and recall")
print("  → Good when you care about both\n")

print("Support = number of samples in each class")

## Part 9: Using All Features

So far we only used 2 features (age, max heart rate). Let's use ALL features for better predictions!

In [ ]:
# Prepare all features
X_all = df.drop(['target'], axis=1).values
y_all = df['target'].values

# Split
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

# Scale features (important for logistic regression!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_all)
X_test_scaled = scaler.transform(X_test_all)

# Train model
log_model_all = LogisticRegression(random_state=42, max_iter=1000)
log_model_all.fit(X_train_scaled, y_train_all)

# Evaluate
train_acc_all = log_model_all.score(X_train_scaled, y_train_all)
test_acc_all = log_model_all.score(X_test_scaled, y_test_all)

print("\n Model Performance (All Features):\n")
print(f"Training Accuracy: {train_acc_all:.3f} ({train_acc_all*100:.1f}%)")
print(f"Testing Accuracy:  {test_acc_all:.3f} ({test_acc_all*100:.1f}%)")

print("\n Comparison:")
print(f"2 features:  {test_accuracy:.3f}")
print(f"All features: {test_acc_all:.3f}")
print(f"Improvement: {(test_acc_all - test_accuracy)*100:.1f} percentage points")

In [ ]:
# Feature importance
feature_names = df.drop(['target'], axis=1).columns
coefficients = log_model_all.coef_[0]

# Sort by absolute value
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
}).sort_values('Abs_Coefficient', ascending=False)

# Plot
plt.figure(figsize=(10, 8))
colors = ['red' if c < 0 else 'green' for c in importance_df['Coefficient']]
plt.barh(importance_df['Feature'], importance_df['Coefficient'], color=colors, alpha=0.7)
plt.xlabel('Coefficient Value', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance (Logistic Regression Coefficients)', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n Top 5 Most Important Features:\n")
for i, row in importance_df.head(5).iterrows():
    direction = "increases" if row['Coefficient'] > 0 else "decreases"
    print(f"{row['Feature']:15} → {direction} disease risk (coef: {row['Coefficient']:+.3f})")

print("\n Positive coefficient = feature increases disease probability")
print(" Negative coefficient = feature decreases disease probability")

## Part 10: Gradient Descent in Logistic Regression

**Remember last class?** We learned gradient descent minimizes loss.

**Question:** Does logistic regression use gradient descent too?

**Answer:** Yes -- But with a different loss function.

### Loss Functions:

**Linear Regression:** Mean Squared Error (MSE)
$$L = \frac{1}{n}\sum(y - \hat{y})^2$$

**Logistic Regression:** Binary Cross-Entropy (Log Loss)
$$L = -\frac{1}{n}\sum[y\log(\hat{y}) + (1-y)\log(1-\hat{y})]$$

where $\hat{y} = \sigma(mx + b)$ is the predicted probability

### Why Different Loss?

- MSE doesn't work well with sigmoid (non-convex, hard to optimize)
- Cross-entropy is convex with sigmoid
- Cross-entropy heavily penalizes confident wrong predictions

**But the algorithm is the same:** Gradient descent

In [ ]:
# Visualize the loss functions
y_true = 1  # Actual label is 1
y_pred = np.linspace(0.01, 0.99, 100)  # Predicted probabilities

# Binary cross-entropy for y=1
log_loss = -np.log(y_pred)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(y_pred, log_loss, 'b-', linewidth=3)
plt.xlabel('Predicted Probability', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Log Loss When True Label = 1', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axvline(x=1.0, color='green', linestyle='--', label='Perfect Prediction')
plt.axvline(x=0.5, color='orange', linestyle='--', label='Uncertain (50/50)')
plt.legend()

# Show both cases
plt.subplot(1, 2, 2)
y_pred_full = np.linspace(0.01, 0.99, 100)
loss_y1 = -np.log(y_pred_full)  # When true label = 1
loss_y0 = -np.log(1 - y_pred_full)  # When true label = 0

plt.plot(y_pred_full, loss_y1, 'b-', linewidth=2, label='True Label = 1')
plt.plot(y_pred_full, loss_y0, 'r-', linewidth=2, label='True Label = 0')
plt.xlabel('Predicted Probability of Class 1', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Log Loss for Both Classes', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim(0, 5)

plt.tight_layout()
plt.show()

print("\n Key Observations:\n")
print("1. Confident correct predictions → low loss ")
print("2. Uncertain predictions (p≈0.5) → moderate loss")
print("3. Confident wrong predictions → very high loss ")
print("\nThis encourages the model to be both accurate AND confident")

## Part 11: Summary & Key Takeaways

### What We Learned Today:

1. **Classification vs Regression**
   - Regression: Predict numbers
   - Classification: Predict categories

2. **Why Not Linear Regression?**
   - Unbounded outputs
   - Can't interpret as probabilities

3. **Logistic Regression Solution**
   - Sigmoid function squeezes output to [0,1]
   - Output = probability of positive class
   - Decision boundary at P=0.5

4. **New Evaluation Metrics**
   - Accuracy (but not always enough!)
   - Confusion Matrix
   - Precision, Recall, F1-Score

5. **Still Uses Gradient Descent!**
   - Same algorithm, different loss function
   - Cross-entropy instead of MSE

### Common Pitfalls:

 **Mistake 1:** Using accuracy when classes are imbalanced
 **Fix:** Look at precision, recall, F1-score

 **Mistake 2:** Forgetting to scale features
 **Fix:** Always use StandardScaler for logistic regression

 **Mistake 3:** Treating probabilities as hard decisions
 **Fix:** Remember you can adjust the threshold (doesn't have to be 0.5)

### What's Next?

- Cross-validation and regularization for classification
- Decision trees and random forests
- **Later:** Multi-class classification (more than 2 categories)
- **pyMAISE:** Will use logistic regression and more advanced classifiers

## Practice Problems

### Problem 1: Threshold Tuning
The default decision threshold is 0.5. What happens if you change it to 0.3 or 0.7? Plot how precision and recall change with different thresholds.

### Problem 2: Feature Selection
Try building a model with only the top 3 most important features. How does it compare to using all features?

### Problem 3: False Negatives Matter
In medical diagnosis, missing a disease (False Negative) is often worse than a false alarm (False Positive). How would you adjust your model to minimize false negatives, even if it means more false positives?

### Problem 4: Class Imbalance
What if 90% of patients are healthy and only 10% have disease? Try creating an imbalanced dataset and see what happens to your metrics. How does accuracy become misleading?

### Problem 5: Sigmoid from Scratch
Implement logistic regression from scratch using gradient descent (like we did for linear regression). The main difference is using the sigmoid function and cross-entropy loss.

### Challenge: Multi-class
Extend the problem to predict 3 levels of heart disease risk (low/medium/high) instead of just binary. How do the metrics change?

In [ ]:
# Space for practice problems

